# From a PDF to structured CONSORT data

### A hands-on Flowde demo

Follow four smoking-cessation studies from their published PDFs to structured
participant-flow data.

**You will:** extract figures with Paddle → correct orientation → identify
CONSORT diagrams → read nodes, labels, arrows and other text → compare the
results with manually annotated ground truth.

**Before you start**

- Sign in to Google Colab with your Google account. A standard CPU runtime is
  sufficient.
- Have the Azure endpoint and API key supplied by the workshop organiser.
- Work down the notebook, pressing the play button beside each code cell. Wait
  for each cell to finish.
- Allow about 10–15 minutes; the first setup downloads software and the Paddle
  model. The time depends on the assigned computer and Azure response times.

All model answers in this notebook come from your live run. The reference
annotations are used only in the final evaluation. The included PDFs and figures
are openly licensed;
[article credits and licences](https://github.com/EPPI-Centre/Flowde/tree/main/data/consort-demo)
are supplied with the dataset.


## 1. Prepare the demo

### Install Flowde

This notebook runs in Google Colab. Run the cell below to install Flowde
directly from GitHub, with support for OpenAI, Paddle image extraction and
benchmarking.

PaddlePaddle 3.2.2 avoids an open CPU compatibility issue in PaddlePaddle 3.3.1.
Flowde installs from the latest code on GitHub.


In [ ]:
%pip install -q "flowde[openai,paddle,benchmark] @ git+https://github.com/EPPI-Centre/Flowde.git" "paddlepaddle==3.2.2"

### Download the example data

Download the four PDFs and their reference annotations from the repository.
`DATA_DIR` points to the example dataset. This download is only needed for the
demo; in your own projects, supply your PDF directory to the extraction step.


In [ ]:
!wget -q https://github.com/EPPI-Centre/Flowde/archive/refs/heads/main.zip -O /content/Flowde.zip
!unzip -q -o /content/Flowde.zip "Flowde-main/data/consort-demo/*" -d /content

from pathlib import Path

DATA_DIR = Path("/content/Flowde-main/data/consort-demo")

### Connect to Azure and choose your settings

Paste the endpoint and key when asked.

For this demo, we will use GPT-5.6 Luna through Azure. We could alternatively
use any model from OpenAI, Gemini or Anthropic. Better results will be obtained
with more intelligent models.

`PDF_DIR` should be set to the folder containing the PDFs to process.

`RUN_DIR` is the folder where Flowde will save your results. Repeating a
classification or parsing cell overwrites that step's previous results. To
keep earlier results when experimenting, choose a new folder name, such as
`/content/flowde-results-low`.


In [ ]:
import os
import time
from getpass import getpass

from flowde.api_utils.openai_utils import check_openai_connection

os.environ["AZURE_API_BASE"] = input("Paste the workshop Azure endpoint: ").strip()
os.environ["AZURE_API_KEY"] = getpass("Paste the workshop Azure API key: ").strip()

MODEL = "gpt-5.6-luna"
REASONING_EFFORT = "medium"  # Try "low" to compare speed and accuracy.
PDF_DIR = DATA_DIR / "pdfs"
RUN_DIR = Path("/content/flowde-results")
RUN_DIR.mkdir(parents=True, exist_ok=True)
started_at = time.perf_counter()

check_openai_connection(model=MODEL, effort=REASONING_EFFORT, from_azure=True)
print("Azure connection successful.")

## 2. Extract images from the PDFs

Flowde reads the PDFs in `PDF_DIR`. Paddle examines every page and finds figure
regions. Flowde saves each figure as a PNG in `EXTRACTED_DIR`. Some figures will
be CONSORT diagrams; others will be graphs or other illustrations.

The first call downloads the image extraction model. Extraction takes several
minutes on Colab CPUs and can run much faster on a GPU. The model can process
2,500 pages per minute on an A100.


In [ ]:
from html import escape

from IPython.display import HTML, display
from PIL import Image

from flowde.extract_imgs import extract_imgs
from flowde.extract_fns.paddle_layout_detect_extraction import (
    make_paddle_layout_extract_fn,
)


def show_images(paths, width=340):
    for path in paths:
        display(HTML(f"<strong>{escape(path.name)}</strong>"))
        with Image.open(path) as image:
            preview = image.copy()
            preview.thumbnail((width, 460))
            display(preview)


EXTRACTED_DIR = RUN_DIR / "extracted"
extract_fn = make_paddle_layout_extract_fn(device="cpu", cpu_threads=1, dpi=144)

extraction_started = time.perf_counter()
extract_imgs(
    pdf_dir=PDF_DIR,
    save_dir=EXTRACTED_DIR,
    extract_fn=extract_fn,
    n_jobs=1,
    on_existing="overwrite",
)
extracted_paths = sorted(EXTRACTED_DIR.glob("*.png"))
print(
    f"Extracted {len(extracted_paths)} images in {time.perf_counter() - extraction_started:.0f} seconds."
)

show_images(extracted_paths)

## 3. Correct image orientation

A diagram may appear sideways in the PDF. Luna chooses the clockwise rotation
needed to make the text upright, and Flowde creates a corrected copy. The
original extracted image is preserved.


In [ ]:
from flowde.classify_fns.classify_types import RotationClassification
from flowde.classify_fns.openai_classify_fn import make_openai_classify_fn
from flowde.rotate_imgs import INPUT_TEXT as ROTATION_PROMPT, rotate_imgs

ROTATION_DIR = RUN_DIR / "rotation"
ROTATED_DIR = ROTATION_DIR / "rotated_images"
rotation_fn = make_openai_classify_fn(
    input_text=ROTATION_PROMPT,
    model=MODEL,
    effort=REASONING_EFFORT,
    result_structure=RotationClassification,
    from_azure=True,
)
angles = rotate_imgs(
    classify_fn=rotation_fn,
    img_dir=EXTRACTED_DIR,
    save_dir=ROTATION_DIR,
    on_existing="overwrite",
    n_jobs=2,
)

for path, angle in zip(extracted_paths, angles, strict=True):
    print(f"{path.name}: {angle} degrees clockwise")
changed = [path for path, angle in zip(extracted_paths, angles, strict=True) if angle]
if changed:
    for path in changed:
        display(HTML("<h4>Before correction</h4>"))
        show_images([path])
        display(HTML("<h4>After correction</h4>"))
        show_images([ROTATED_DIR / path.name])
else:
    print("Luna judged all selected images to be upright.")

## 4. Keep the CONSORT diagrams

Flowde asks Luna which figures show participant flow. A label of 1 means
CONSORT; 0 means another kind of figure. Only the figures labelled 1 continue to
parsing.

These are model decisions. An incorrect decision will remain visible in the
evaluation.


In [ ]:
from flowde.classify_fns.classify_types import ConsortClassification
from flowde.classify_imgs import classify_imgs

CLASSIFICATION_PROMPT = (
    "Classify the image as a CONSORT image with label 1.\n"
    "If the image is not a CONSORT image, classify it with label 0."
)
classification_fn = make_openai_classify_fn(
    input_text=CLASSIFICATION_PROMPT,
    model=MODEL,
    effort=REASONING_EFFORT,
    result_structure=ConsortClassification,
    from_azure=True,
)
CLASSIFICATION_DIR = RUN_DIR / "classification"
labels = classify_imgs(
    classify_fn=classification_fn,
    img_dir=ROTATED_DIR,
    save_dir=CLASSIFICATION_DIR,
    positive_classes={1},
    n_jobs=2,
    on_existing="overwrite",
)
CONSORT_DIR = CLASSIFICATION_DIR / "positive_images"
consort_paths = sorted(CONSORT_DIR.glob("*.png"))
print(f"Luna selected {len(consort_paths)} of {len(extracted_paths)} images.")
show_images(consort_paths)
if not consort_paths:
    raise RuntimeError(
        "No CONSORT diagrams were selected. Review the classification results before continuing."
    )

## 5. Read each diagram in four parts

Flowde's existing CONSORT prompts separate the job into four steps:

1. **Nodes:** the text in each box, with a number identifying each box.
2. **Labels:** headings or other text that apply to particular boxes.
3. **Flow:** the arrows connecting the boxes.
4. **Additional text:** captions, notes and other diagram text.

Later steps receive the node numbers and the relevant earlier results. The
reference annotations are not sent to Luna. Each step saves one JSON file per
image.


In [ ]:
from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn
from flowde.prompts.consort import consort_nodes_prompt
from flowde.prompts.consort_labels import consort_labels_prompt
from flowde.prompts.consort_flow import consort_flow_prompt
from flowde.prompts.consort_add_text import consort_add_text_prompt

PARSED_DIR = RUN_DIR / "parsed"


def parse_part(prompt, part, **previous_parts):
    parser = make_openai_parse_fn(
        input_text=prompt,
        model=MODEL,
        effort=REASONING_EFFORT,
        parts_to_parse={part},
        from_azure=True,
    )
    results = parse_imgs(
        parse_fn=parser,
        img_dir=CONSORT_DIR,
        save_dir=PARSED_DIR / part,
        n_jobs=2,
        on_existing="overwrite",
        **previous_parts,
    )
    print(f"{part}: saved results for {len(results)} images.")
    return results

### 5a. Nodes

First, read the box text and assign node numbers.


In [ ]:
node_results = parse_part(consort_nodes_prompt, "node_text")

### 5b. Labels

Next, identify headings and labels. The existing node results tell Luna which
node number each label belongs to.


In [ ]:
label_results = parse_part(
    consort_labels_prompt,
    "labels",
    nodes_dir=PARSED_DIR / "node_text",
)

### 5c. Flow

Now, follow the arrows. Each `points_to` list records the node numbers reached
from a box.


In [ ]:
flow_results = parse_part(
    consort_flow_prompt,
    "flow",
    nodes_dir=PARSED_DIR / "node_text",
    labels_dir=PARSED_DIR / "labels",
)

### 5d. Additional text

Finally, read any remaining captions or notes, with the preceding results
supplied so the same text is not deliberately assigned twice.


In [ ]:
additional_results = parse_part(
    consort_add_text_prompt,
    "additional_texts",
    nodes_dir=PARSED_DIR / "node_text",
    labels_dir=PARSED_DIR / "labels",
    flow_dir=PARSED_DIR / "flow",
)

### Inspect a complete result

The four parts are combined into one JSON file per diagram in `parsed/combined/`.
The first combined result is shown below. The separate parts remain available.


In [ ]:
from IPython.display import JSON

from flowde.parsing_fns.parsing_types import build_partial_flowcharts

node_paths = sorted((PARSED_DIR / "node_text").glob("*.json"))
complete_results = build_partial_flowcharts(
    nodes_paths=node_paths,
    labels_paths=sorted((PARSED_DIR / "labels").glob("*.json")),
    flow_paths=sorted((PARSED_DIR / "flow").glob("*.json")),
    additional_texts_paths=sorted((PARSED_DIR / "additional_texts").glob("*.json")),
)

COMBINED_DIR = PARSED_DIR / "combined"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
for previous_result in COMBINED_DIR.glob("*.json"):
    previous_result.unlink()
for node_path, result in zip(node_paths, complete_results, strict=True):
    (COMBINED_DIR / node_path.name).write_text(
        result.model_dump_json(indent=2) + "\n", encoding="utf-8"
    )

show_images(consort_paths[:1], width=600)
display(JSON(complete_results[0].model_dump(), expanded=True))

## 6. Evaluate the results

The reference dataset contains a manually annotated CONSORT diagram for each
selected PDF. First, check which diagrams the classifier found or missed. Then
use Flowde's parsing benchmark to compare the recognised reference diagrams with
their annotations.

- **Node text edits:** character insertions, deletions and substitutions needed
  to match the reference node text. Lower is better; 0 is an exact match after
  whitespace and Unicode normalisation.
- **Flow Jaccard:** overlap between the predicted arrows and reference arrows
  after matching the nodes. Higher is better; 1 is an exact match.

Parsing scores cover recognised reference diagrams only. Missed diagrams and
extra selections are reported separately, so a good parsing score does not hide
a classification error. References may contain several accepted interpretations;
Flowde matches against the available options.


In [ ]:
import json
import shutil

from flowde.benchmarks.parsing.parsing_bench import ParsingBenchmark
from flowde.benchmarks.parsing.text_distance_fns.levenshtein_fn import (
    levenshtein_with_nfc_and_space_normalisation,
)

papers = json.loads((DATA_DIR / "manifest.json").read_text())["papers"]
true_names = {Path(name).stem for paper in papers for name in paper["consort_images"]}
predicted_names = {p.stem for p in consort_paths}
matched_names = true_names & predicted_names
missed_names = sorted(true_names - predicted_names)
extra_names = sorted(predicted_names - true_names)

print(f"Reference CONSORTs found: {len(matched_names)} / {len(true_names)}")
print("Missed CONSORTs:", missed_names or "None")
print("Other figures selected as CONSORT:", extra_names or "None")

# Evaluate only diagrams for which both a prediction and a reference exist.
# Keep this subset separate; all original live predictions remain in parsed/.
parts = ["nodes", "labels", "flow", "additional_texts"]
benchmark_root = RUN_DIR / "benchmark"
if benchmark_root.exists():
    shutil.rmtree(benchmark_root)
for part in parts:
    for name in true_names:
        target = benchmark_root / "truth" / part / f"{name}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(DATA_DIR / "ground-truth" / part / target.name, target)
    for name in matched_names:
        target = benchmark_root / "predictions" / part / f"{name}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        prediction_part = "node_text" if part == "nodes" else part
        shutil.copyfile(PARSED_DIR / prediction_part / target.name, target)

summary = {
    "reference_diagrams": len(true_names),
    "recognised_reference_diagrams": len(matched_names),
    "missed_diagrams": missed_names,
    "extra_selections": extra_names,
    "model": MODEL,
    "reasoning_effort": REASONING_EFFORT,
}
if matched_names:
    benchmark = ParsingBenchmark(
        pred_diagrams_dir=[benchmark_root / "predictions" / p for p in parts],
        distance_fn=levenshtein_with_nfc_and_space_normalisation,
        true_nodes_dir=benchmark_root / "truth/nodes",
        true_labels_dir=benchmark_root / "truth/labels",
        true_flow_dir=benchmark_root / "truth/flow",
        true_additional_texts_dir=benchmark_root / "truth/additional_texts",
        expected_num_diagrams=len(true_names),
        allow_missing_pred_diagrams=True,
    )
    summary["node_text_edits"] = benchmark.total_node_text_cost()
    summary["mean_flow_jaccard"] = benchmark.avg_flow_jaccard()
    print(f"Node text edits: {summary['node_text_edits']}")
    print(f"Mean flow Jaccard: {summary['mean_flow_jaccard']:.3f}")
else:
    print("No recognised reference diagrams are available for parsing scores.")

summary["elapsed_seconds_after_setup"] = round(time.perf_counter() - started_at, 1)
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(
    f"Elapsed time after setup: {summary['elapsed_seconds_after_setup'] / 60:.1f} minutes."
)

## Take your results away

The ZIP contains your extracted and corrected images, classification decisions,
separate parsed parts, combined JSON files in `parsed/combined/`, and the
evaluation summary. Download the ZIP before closing Colab: the runtime's files
are temporary.

To experiment, change the reasoning setting or `PDF_DIR` near the top and run
the cells again. Choose a new `RUN_DIR` folder name to keep earlier results.
Any repeated live model calls use the Azure account's quota.


In [ ]:
from google.colab import files

archive_path = shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
files.download(archive_path)